# Phase 2: Instruction Finetuning on OpenAssistant-Guanaco

This notebook handles instruction finetuning of the pretrained TRM model.

## Training Goals
- **Dataset**: OpenAssistant-Guanaco (~10k conversations)
- **Base Model**: Checkpoint from Phase 1 (pretrained TRM)
- **Training**: 3-5 epochs with lower learning rate
- **Hardware**: 2x A40 GPUs
- **Duration**: ~4-8 hours

## Key Features
- Load pretrained checkpoint from Phase 1
- Format conversations as "### Human: ... ### Assistant: ..."
- Lower learning rate for finetuning
- More frequent checkpointing (every 500 steps)
- Evaluation every 200 steps

## Setup and Imports

In [ ]:
import sys
import os
from pathlib import Path

# Add src to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import torch
from transformers import GPT2Tokenizer
from accelerate import Accelerator

from src.config import ModelConfig, InstructionTuningConfig, WandbConfig
from src.model import HybridTRM
from src.data import get_instruction_dataloader, create_validation_dataset
from src.training import train, setup_scheduler
from src.utils import (
    enable_tf32,
    setup_wandb,
    load_checkpoint,
    get_latest_checkpoint,
    print_model_info,
    finish_wandb,
)

print("✓ Imports successful")

## Configuration

In [ ]:
# Model configuration (same as pretraining)
model_config = ModelConfig(
    base_model="gpt2",
    n_latents=64,
    n_sup=8,
    t_loops=3,
    seq_len_x=512,
    seq_len_y=512,
    use_rope=True,
    gradient_checkpointing=True,
)

# Instruction tuning configuration
train_config = InstructionTuningConfig(
    dataset_name="timdettmers/openassistant-guanaco",
    dataset_format="openassistant",
    learning_rate=1e-5,  # Lower LR for finetuning
    weight_decay=0.1,
    warmup_steps=100,
    num_epochs=3,
    batch_size_per_gpu=16,  # Smaller batch for longer sequences
    gradient_accumulation_steps=2,  # Effective batch = 32 per GPU
    max_grad_norm=1.0,
    checkpoint_every=500,
    eval_every=200,
    log_every=50,
    eval_samples=500,
    mixed_precision="bf16",
    seed=42,
)

# WandB configuration
wandb_config = WandbConfig(
    project="recursive-reasoning-trm",
    name="finetune-gpt2-small-guanaco",
    tags=["instruction-tuning", "gpt2-small", "openassistant-guanaco"],
    notes="Instruction finetuning GPT-2 Small with 64 latent tokens on OpenAssistant-Guanaco",
    mode="online",
)

# Paths
PRETRAIN_CHECKPOINT = project_root / "checkpoints" / "pretrain" / "checkpoint_final.pt"
CHECKPOINT_DIR = project_root / "checkpoints" / "finetune"
LOG_DIR = project_root / "logs" / "finetune"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# Resume from finetuning checkpoint?
RESUME_FINETUNE = False  # Set to True to resume finetuning (not pretraining!)

print("✓ Configuration loaded")
print(f"Pretrain checkpoint: {PRETRAIN_CHECKPOINT}")
print(f"Checkpoint dir: {CHECKPOINT_DIR}")
print(f"Log dir: {LOG_DIR}")

## Initialize Accelerator

In [ ]:
# Enable TF32
enable_tf32()

# Initialize Accelerator
accelerator = Accelerator(
    mixed_precision=train_config.mixed_precision,
    gradient_accumulation_steps=train_config.gradient_accumulation_steps,
    log_with="wandb" if wandb_config.mode == "online" else None,
)

print(f"✓ Accelerator initialized")
print(f"  Device: {accelerator.device}")
print(f"  Num processes: {accelerator.num_processes}")

## Initialize Model and Load Pretrained Checkpoint

In [ ]:
# Set seed
torch.manual_seed(train_config.seed)

# Initialize model
model = HybridTRM(model_config)

if accelerator.is_main_process:
    print_model_info(model)

print("✓ Model initialized")

## Initialize Tokenizer and Dataloaders

In [ ]:
# Initialize tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# Training dataloader
train_dataloader = get_instruction_dataloader(
    config=train_config,
    model_config=model_config,
    tokenizer=tokenizer,
    split="train",
)

# Validation dataloader
print("Creating validation dataset (sampling 500 examples)...")
eval_examples = create_validation_dataset(
    train_dataloader,
    num_samples=train_config.eval_samples,
)

eval_dataloader = torch.utils.data.DataLoader(
    eval_examples,
    batch_size=train_config.batch_size_per_gpu,
    shuffle=False,
)

print("✓ Dataloaders initialized")
print(f"  Eval samples: {len(eval_examples)}")

## Initialize Optimizer and Scheduler

In [ ]:
# Optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=train_config.learning_rate,
    weight_decay=train_config.weight_decay,
    betas=(train_config.beta1, train_config.beta2),
    eps=train_config.eps,
)

# Estimate training steps (rough estimate for instruction tuning)
# OpenAssistant-Guanaco has ~10k examples
estimated_examples = 10000
batch_size = train_config.batch_size_per_gpu
num_gpus = accelerator.num_processes
grad_accum = train_config.gradient_accumulation_steps
steps_per_epoch = estimated_examples // (batch_size * num_gpus * grad_accum)
num_training_steps = steps_per_epoch * train_config.num_epochs

# Learning rate scheduler
scheduler = setup_scheduler(
    optimizer=optimizer,
    train_config=train_config,
    num_training_steps=num_training_steps,
)

print("✓ Optimizer and scheduler initialized")
print(f"  Estimated steps per epoch: {steps_per_epoch}")
print(f"  Total training steps: {num_training_steps}")

## Prepare for Distributed Training

In [ ]:
# Prepare with Accelerator
model, optimizer, train_dataloader, eval_dataloader, scheduler = accelerator.prepare(
    model, optimizer, train_dataloader, eval_dataloader, scheduler
)

print("✓ Model prepared for distributed training")

## Load Pretrained Checkpoint

In [ ]:
start_step = 0
start_epoch = 0

if RESUME_FINETUNE:
    # Resume from finetuning checkpoint
    latest_checkpoint = get_latest_checkpoint(str(CHECKPOINT_DIR))
    if latest_checkpoint:
        print(f"Resuming finetuning from: {latest_checkpoint}")
        metadata = load_checkpoint(
            path=latest_checkpoint,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            accelerator=accelerator,
        )
        start_step = metadata["step"]
        start_epoch = metadata["epoch"]
    else:
        print("No finetuning checkpoint found, loading pretrained model...")
        metadata = load_checkpoint(
            path=str(PRETRAIN_CHECKPOINT),
            model=model,
            optimizer=None,  # Don't load optimizer from pretrain
            scheduler=None,
            accelerator=accelerator,
            load_optimizer=False,
        )
else:
    # Load pretrained model (fresh start for finetuning)
    print(f"Loading pretrained model from: {PRETRAIN_CHECKPOINT}")
    metadata = load_checkpoint(
        path=str(PRETRAIN_CHECKPOINT),
        model=model,
        optimizer=None,
        scheduler=None,
        accelerator=accelerator,
        load_optimizer=False,
    )
    print("✓ Pretrained model loaded (starting finetuning from scratch)")

## Initialize Weights & Biases

In [ ]:
# Setup WandB
setup_wandb(
    config=wandb_config,
    train_config=train_config,
    model_config=model_config,
    accelerator=accelerator,
)

## Training Loop

In [ ]:
# Run training
train(
    model=model,
    train_dataloader=train_dataloader,
    eval_dataloader=eval_dataloader,
    optimizer=optimizer,
    scheduler=scheduler,
    accelerator=accelerator,
    model_config=model_config,
    train_config=train_config,
    checkpoint_dir=str(CHECKPOINT_DIR),
    log_dir=str(LOG_DIR),
    start_step=start_step,
    start_epoch=start_epoch,
)

## Cleanup

In [ ]:
# Finish WandB
finish_wandb(accelerator)

print("\n✓ Instruction finetuning completed!")
print(f"Final checkpoint: {CHECKPOINT_DIR}/checkpoint_final.pt")

## Interactive Testing: Generate Responses

Test the finetuned model with conversational prompts.

In [ ]:
# Helper function for conversational generation
def generate_response(prompt_text: str, max_tokens: int = 100, temperature: float = 0.8):
    """Generate a response to a human prompt."""
    model.eval()
    
    # Format prompt
    formatted_prompt = f"### Human: {prompt_text}\n### Assistant:"
    
    # Tokenize
    input_ids = tokenizer.encode(formatted_prompt, return_tensors="pt").to(accelerator.device)
    
    # Generate
    with torch.no_grad():
        generated_ids = model.generate(
            x_ids=input_ids,
            max_new_tokens=max_tokens,
            temperature=temperature,
            top_k=50,
        )
    
    # Decode
    response = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    
    return response

In [ ]:
# Test with sample prompts
test_prompts = [
    "What is the capital of France?",
    "Explain quantum computing in simple terms.",
    "Write a haiku about machine learning.",
    "How do I make chocolate chip cookies?",
]

for prompt in test_prompts:
    print("=" * 70)
    print(f"Human: {prompt}")
    response = generate_response(prompt, max_tokens=100, temperature=0.7)
    print(f"Assistant: {response}")
    print()

In [ ]:
# Interactive mode (uncomment to use)
# while True:
#     user_input = input("\nYou: ")
#     if user_input.lower() in ["quit", "exit", "q"]:
#         break
#     response = generate_response(user_input)
#     print(f"Assistant: {response}")